# 📦 Step 1 — Download Qwen3.6 Model & Create Kaggle Dataset

This notebook is run **once**. It downloads the Qwen3.6-35B-A3B GGUF model files from HuggingFace directly onto Kaggle's servers, then packages them into a **permanent private Kaggle Dataset**.

Once this dataset exists, the main server notebook (`2_run_server.ipynb`) can mount it instantly on every run — no re-downloading ever again.

---

### What gets downloaded

| File | Size | Purpose |
|---|---|---|
| `Qwen3.6-35B-A3B-UD-Q4_K_XL.gguf` | ~22.4 GB | The main language model (4-bit quantized) |
| `mmproj-F16.gguf` | ~1 GB | The vision projector — required for multimodal (image) support |

### Why two files?

Qwen3.6 is a **multimodal** model: it understands both text and images. The vision projector (`mmproj`) is a separate file that bridges the image encoder to the language model. `llama.cpp` loads both at startup via the `--mmproj` argument.

> ⚠️ **Note:** This is exactly why Ollama cannot run Qwen3.6 — it does not support separate mmproj files. `llama.cpp` is the correct backend for this model.

---

### Prerequisites — Before running this notebook

1. In the Kaggle notebook settings (right panel), make sure **Internet** is set to **ON**
2. No GPU is needed for this notebook — CPU is fine
3. Your Kaggle account must have enough free storage (Kaggle gives 100 GB free)

## ⚙️ Cell 1 — Install dependencies

We install two packages:
- `huggingface_hub` — the official HuggingFace download library
- `hf_transfer` — a fast Rust-based transfer backend that makes large downloads **3-5x faster**

In [ ]:
%%capture
!pip install -q huggingface_hub hf_transfer

## ⚙️ Cell 2 — Enable fast transfer & define paths

Setting `HF_HUB_ENABLE_HF_TRANSFER=1` activates the fast Rust backend automatically.

We download into `/tmp/model/` instead of `/kaggle/working/`.

> **Why `/tmp` and not `/kaggle/working`?**  
> Kaggle enforces a **20 GB quota** on `/kaggle/working/` (the output directory). Our model alone weighs 22.4 GB — it doesn't fit.  
> `/tmp` shares the full disk (≈73 GB) with no per-folder quota, so the download fits comfortably.  
> We don't need the files to survive the session — once the Kaggle Dataset is created, the local copy is no longer needed.

In [ ]:
import os
import json
from pathlib import Path

# Enable fast transfer (3-5x speedup on large files)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# ── Configuration ──────────────────────────────────────────────────────────────
REPO_ID      = "unsloth/Qwen3.6-35B-A3B-GGUF"   # HuggingFace model repo
QUANT        = "UD-Q4_K_XL"                       # Quantization variant
DOWNLOAD_DIR = Path("/tmp/model")                  # Where files land
# ⚠️  We use /tmp and not /kaggle/working because Kaggle enforces a 20 GB
# quota on /kaggle/working — not enough for a 22.4 GB model file.
# /tmp shares the full ~73 GB disk with no per-folder limit.

# Dataset name that will appear on your Kaggle account
# You can change this slug — it must be lowercase with hyphens only
DATASET_SLUG = "qwen36-35b-a3b-gguf"
# ───────────────────────────────────────────────────────────────────────────────

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
print(f"📁 Download directory: {DOWNLOAD_DIR}")
print(f"🤗 Source repo      : {REPO_ID}")
print(f"🗜️  Quantization     : {QUANT}")

## 📥 Cell 3 — Download model files from HuggingFace

We use `hf_hub_download` with glob filters to grab only the two files we need:
- `*UD-Q4_K_XL*` → the main model
- `*mmproj-F16*` → the vision projector

This avoids downloading the entire repo (which includes other quants we don't need).

> ⏳ Estimated download time: **15–30 minutes** depending on Kaggle's connection speed.

In [ ]:
from huggingface_hub import snapshot_download

print("🚀 Starting download from HuggingFace...")
print("   This will take 15–30 minutes. You can monitor progress below.\n")

snapshot_download(
    repo_id   = REPO_ID,
    repo_type = "model",
    local_dir = str(DOWNLOAD_DIR),
    allow_patterns = [
        f"*{QUANT}*",   # Main model file (~22.4 GB)
        "*mmproj-F16*"  # Vision projector (~1 GB)
    ],
)

print("\n✅ Download complete.")

## 🔍 Cell 4 — Verify downloaded files

Before creating the dataset, we verify that both files are present and have a reasonable size.

Expected sizes:
- Main model: ~22 GB
- mmproj: ~1 GB

In [ ]:
print("📋 Downloaded files:\n")

total_size = 0
found_model  = False
found_mmproj = False

for f in sorted(DOWNLOAD_DIR.rglob("*.gguf")):
    size_gb = f.stat().st_size / (1024 ** 3)
    total_size += size_gb
    tag = ""

    if QUANT in f.name:
        found_model = True
        tag = "← main model"
    if "mmproj" in f.name:
        found_mmproj = True
        tag = "← vision projector"

    print(f"  {f.name:<55} {size_gb:6.2f} GB  {tag}")

print(f"\n  Total: {total_size:.2f} GB")
print()

# Sanity checks
assert found_model,  "❌ Main model file not found! Check the download."
assert found_mmproj, "❌ mmproj file not found! Multimodal will not work."
print("✅ Both files verified — ready to create the Kaggle Dataset.")

## 📦 Cell 5 — Create the Kaggle Dataset

Kaggle requires a `dataset-metadata.json` file alongside the data files. We generate it automatically using your Kaggle username (read from the environment — Kaggle injects it automatically in notebooks).

Then we run `kaggle datasets create` to publish the dataset to your account as **private**.

> This step uploads the files to Kaggle's storage. It may take a few minutes.

In [ ]:
import subprocess

# ── Your Kaggle username ────────────────────────────────────────────────────────
# Find it at: https://www.kaggle.com/settings  (under 'Username')
KAGGLE_USERNAME = ""   # ← fill this in, e.g. "johndoe"
# ───────────────────────────────────────────────────────────────────────────────

# Fallback: try environment variable (works in interactive Kaggle sessions)
if not KAGGLE_USERNAME:
    KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "")

if not KAGGLE_USERNAME:
    raise ValueError(
        "KAGGLE_USERNAME is empty.\n"
        "Please set it at the top of this cell: KAGGLE_USERNAME = 'your_username'"
    )

kaggle_username = KAGGLE_USERNAME

print(f"👤 Kaggle username: {kaggle_username}")
print(f"📦 Dataset will be created as: {kaggle_username}/{DATASET_SLUG}")
print()

# ── Write dataset-metadata.json ────────────────────────────────────────────────
metadata = {
    "title"   : "Qwen3.6-35B-A3B GGUF",
    "id"      : f"{kaggle_username}/{DATASET_SLUG}",
    "licenses": [{"name": "apache-2.0"}],
    "isPrivate": True
}

metadata_path = DOWNLOAD_DIR / "dataset-metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("📝 dataset-metadata.json created.")

# ── Push dataset to Kaggle ─────────────────────────────────────────────────────
# --dir-mode skip : upload files as-is, no compression.
# We avoid --dir-mode zip because zipping a 22 GB file needs extra disk space
# and the GGUF format is already an optimized binary — compression gains nothing.
print("⬆️  Uploading to Kaggle (this may take several minutes)...\n")

result = subprocess.run(
    ["kaggle", "datasets", "create", "-p", str(DOWNLOAD_DIR), "--dir-mode", "skip"],
    capture_output=True, text=True
)

print(result.stdout)
if result.returncode != 0:
    print("⚠️ stderr:", result.stderr)
else:
    print(f"\n✅ Dataset created successfully!")
    print(f"🔗 Access it at: https://www.kaggle.com/datasets/{kaggle_username}/{DATASET_SLUG}")

## ✅ Done!

Your model files are now stored permanently in your Kaggle account.

### What's next?

Open `2_run_server.ipynb` and add your dataset as an input:

1. In the Kaggle notebook settings → **Add data** → search for `qwen36-35b-a3b-gguf`
2. Select your private dataset
3. The files will be available at `/kaggle/input/qwen36-35b-a3b-gguf/`

---

### Paths to remember for the server notebook

```python
MODEL_PATH  = "/kaggle/input/qwen36-35b-a3b-gguf/Qwen3.6-35B-A3B-UD-Q4_K_XL.gguf"
MMPROJ_PATH = "/kaggle/input/qwen36-35b-a3b-gguf/mmproj-F16.gguf"
```